# 加载数据集

图片像素信息在各端的不一致情况。

In [15]:

from torch.utils.data import DataLoader
from torchvision import datasets

full_dataset = datasets.ImageFolder("datasets/train")
print("loaded full dataset...")
print("总样本数:", len(full_dataset))
print("类别:", full_dataset.classes)
print("类别映射:", full_dataset.class_to_idx)



loaded full dataset...
总样本数: 24993
类别: ['Cat', 'Dog']
类别映射: {'Cat': 0, 'Dog': 1}


In [16]:
import time

start = time.time()

img, label = full_dataset[0]

print("读取耗时:", time.time() - start)
print("图片类型:", type(img))
print("图片尺寸:", img.shape if hasattr(img, "shape") else img.size)
print("label:", label)

读取耗时: 0.0019991397857666016
图片类型: <class 'PIL.Image.Image'>
图片尺寸: (500, 375)
label: 0


In [17]:
import matplotlib

print(matplotlib.get_backend())
print(matplotlib.get_backend())

import matplotlib.pyplot as plt

plt.figure()
plt.plot([1, 2, 3, 4])
plt.show()
#
# import matplotlib.pyplot as plt
#
# img, label = full_dataset[0]
#
# print("准备显示")
#
# plt.imshow(img)
# plt.axis("off")
# plt.show()
#
# print("显示完成")

module://matplotlib_inline.backend_inline
module://matplotlib_inline.backend_inline



KeyboardInterrupt



KeyboardInterrupt: 

In [ ]:
import random
import matplotlib.pyplot as plt

# 随机找8个脚标
indices = random.sample(range(len(full_dataset)), 8)
# 定义2*4画布
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
# 循环散入数据
for ax, idx in zip(axes.flat, indices):
    img, label = full_dataset[idx]

    ax.imshow(img)
    ax.set_title(
        f"label={label} ({full_dataset.classes[label]})"
    )
    ax.axis("off")
plt.tight_layout()
plt.show()
print("show random eight picture done ....")

In [ ]:
from torchvision import transforms
import torch

seed = 42
generator = torch.Generator().manual_seed(seed)

train_indices = []
val_indices = []

for label in range(len(full_dataset.classes)):
    # 找到当前类别的所有样本下标
    indices = [
        i for i, target in enumerate(full_dataset.targets)
        if target == label
    ]

    # 随机打乱
    indices = torch.tensor(indices)
    indices = indices[torch.randperm(len(indices), generator=generator)].tolist()

    # 80%训练，20%验证
    split = int(0.8 * len(indices))

    train_indices.extend(indices[:split])
    val_indices.extend(indices[split:])
print("split dataset to train and val done.")

from collections import Counter

train_labels = [full_dataset.targets[i] for i in train_indices]
val_labels = [full_dataset.targets[i] for i in val_indices]

print("训练集:")
print(Counter(train_labels))

print("验证集:")
print(Counter(val_labels))


In [ ]:
from torch.utils.data import Subset

print("begin create dataset...")
# 1、resize修改尺寸了，怎么判断应该改到什么尺寸？如果原图尺寸不够怎么办？
# 答：主要根据两个因素，模型的输入要求/常见的输入尺寸；任务对图像细节的要求。resnet一般都是224*224.如果原图小于这个尺寸，那么像素将会增加，原细节并未增多。这是一个信息量和计算量之间的权衡。
# 2. 怎么判断normalize里面的值，它是固定的么？
# 答： 基本上是固定的，这个数据是ImageNetRGB通道的均值和标准差。因为使用ImageNet的预训练参数，所以也采用对应的均值和标准差。
# 当然也可以自己计算得出：计算方法，先toTensor变成0~1的数值区间，会得到三个，RGB通道各一个。与之对应的就是三个均值，三个标准差。
# 如果使用预训练模型就用人家的。如果从零开始训练，就可以自己算
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# 因为full_dataset没有transform，train和val要用不同的tranform，所以需要单独创建两个单独的ImageFolder，然后再通过脚标索引去使用它
train_full = datasets.ImageFolder(
    "datasets/train",
    transform=train_transform
)
val_full = datasets.ImageFolder(
    "datasets/train",
    transform=val_transform
)
train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)
print("create two datasets done")

In [ ]:
# 创建dataloader
batch_size = 64
print("begin create dataloader")
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # 代表每个epoch都打乱顺序
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)
# 获取一个batchsize数据，然后展示数据的星霜
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print("create dataLoader done")

# 模型选择与创建
这里只有猫狗两个类别，也就是二分类。属于简单分类，先用ResNet18


In [ ]:
import torch
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用 {device} 进行训练")
model = models.resnet18(
    weights=models.ResNet18_Weights)

print("change nn linear")
# 因为这个项目是二分类，所以全连接输出层需要修改（ResNet默认100分类）
from torch import nn

model.fc = nn.Linear(
    model.fc.in_features,  #
    2
)
model = model.to(device)
print("create model done")
print(model.fc)

# 定义损失函数和迭代器


In [ ]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=1e-4
)
print("define loss and optim done")

# 开始训练

In [31]:
import os

running_loss = 0.0
correct = 0
total = 0

# 这里loader已经设置了批量的大小，所以这里相当于next，每次都会取出固定批次数据
print("begin train")
num_epochs = 10

best_val_acc = 0.0
best_model_path = "best_ResNet18.pth"
checkpoint_path = "checkpoint_ResNet18.pth"
start_epoch = 0

if os.path.exists(checkpoint_path):
    # 检查有没有check_point,如果有加载
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    start_epoch = checkpoint["epoch"] + 1
    best_val_acc = checkpoint["best_val_acc"]
    print(f"从{start_epoch}开始训练")

for epoch in range(start_epoch, num_epochs):
    # 设置训练模式
    model.train()
    # 当前 epoch 的统计量
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        #1. 清空上一轮梯度：清除model.parameter.gard的值
        optimizer.zero_grad()

        #2. 前向传播，即开始预测
        outputs = model(images)

        #3. 计算损失loss
        loss = criterion(outputs, labels)

        # 4. 反向传播:从 loss 开始，沿着计算图反向计算每一个参数对 loss 的梯度。
        # loss 是一个 Tensor，autograd 通过计算图记录 loss 与模型参数之间的计算依赖关系。
        # backward() 沿计算图反向计算 loss 对模型参数的梯度，并写入 parameter.grad。
        loss.backward()

        # 5. 更新模型参数:用刚才计算的梯度去更新参数。根据model.parameter.gard的值去更新model.parameter
        optimizer.step()

        # 统计loss
        running_loss += loss.item()

        # 统计准确率
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # 当前 epoch 的平均 loss 和 accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # 设置验证模式
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        # 标识以下操作均不进行梯度计算
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            #1. 前向传播
            outputs = model(images)

            #2.计算损失loss
            loss - criterion(outputs, labels)

            #3. 统计loss
            val_running_loss += loss.item()

            #统计准确率
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    # 统计当前epoch的验证集loss和accuracy
    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "epoch": epoch,
                "model_name": "resnet18",
                "class_to_idx": full_dataset.class_to_idx,

                "model_state_dict": model.state_dict(),

                "best_val_acc": best_val_acc,

                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,

                "batch_size": batch_size,
                "learning_rate": 1e-4,
            },
            best_model_path
        )
        print(f"保存最佳模型 {best_val_acc:.4f}")
        # 保存当前的模型
        torch.save(
            {
                "epoch": epoch,
                "model_name": "resnet18",
                "class_to_idx": full_dataset.class_to_idx,

                "model_state_dict": model.state_dict(),

                "best_val_acc": best_val_acc,

                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,

                "batch_size": batch_size,
                "learning_rate": 1e-4,
            },
            best_model_path
        )

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )


begin train


C:\Users\22035483\.conda\envs\test311\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


保存最佳模型 0.9890
Epoch [1/10] Train Loss: 0.0029 Train Acc: 0.9989 Val Loss: 0.0008 Val Acc: 0.9890
Epoch [2/10] Train Loss: 0.0033 Train Acc: 0.9989 Val Loss: 0.0002 Val Acc: 0.9888
保存最佳模型 0.9892
Epoch [3/10] Train Loss: 0.0027 Train Acc: 0.9992 Val Loss: 0.0000 Val Acc: 0.9892
Epoch [4/10] Train Loss: 0.0025 Train Acc: 0.9992 Val Loss: 0.0001 Val Acc: 0.9890
保存最佳模型 0.9896
Epoch [5/10] Train Loss: 0.0030 Train Acc: 0.9988 Val Loss: 0.0002 Val Acc: 0.9896
Epoch [6/10] Train Loss: 0.0032 Train Acc: 0.9987 Val Loss: 0.0002 Val Acc: 0.9888
Epoch [7/10] Train Loss: 0.0034 Train Acc: 0.9988 Val Loss: 0.0001 Val Acc: 0.9888
Epoch [8/10] Train Loss: 0.0037 Train Acc: 0.9986 Val Loss: 0.0000 Val Acc: 0.9892
Epoch [9/10] Train Loss: 0.0028 Train Acc: 0.9989 Val Loss: 0.0002 Val Acc: 0.9892
Epoch [10/10] Train Loss: 0.0029 Train Acc: 0.9991 Val Loss: 0.0006 Val Acc: 0.9894


# 加载模型，开始推理

In [35]:
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)
model.fc = nn.Linear(
    model.fc.in_features,
    2
)
model = model.to(device)
print("加载模型")
checkpoint = torch.load(
    best_model_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
#加载分类名称和对应索引
class_to_idx = checkpoint["class_to_idx"]

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}
print("当前加载的类别信息为")
print(idx_to_class)

print("模型加载成功")
print(f"最佳验证准确率: ",checkpoint["best_val_acc"])

# 加载和训练过程中 val相同的transform
from torchvision import transforms

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def predict(image_paths, model, transform, idx_to_class, device):
    """
    对一批没有标签的图片进行预测。

    参数：
        image_paths: 图片路径列表
        model: 已加载好的模型
        transform: 推理阶段的 transform
        idx_to_class: 类别索引 -> 类别名称
        device: cpu / cuda

    返回：
        results: 每张图片的预测结果
    """

    model.eval()

    results = []

    # =========================
    # 1. 读取并预处理所有图片
    # =========================

    tensors = []

    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")

        tensor = transform(image)

        tensors.append(tensor)

    # =========================
    # 2. 拼成 Batch
    # =========================

    batch = torch.stack(tensors)

    batch = batch.to(device)

    # =========================
    # 3. 批量推理
    # =========================

    with torch.no_grad():

        outputs = model(batch)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidences, predictions = torch.max(
            probabilities,
            dim=1
        )

    # =========================
    # 4. 整理结果
    # =========================

    for image_path, prediction, confidence in zip(
            image_paths,
            predictions,
            confidences
    ):
        prediction = prediction.item()
        confidence = confidence.item()

        results.append({
            "path": image_path,
            "class": idx_to_class[prediction],
            "confidence": confidence
        })

    return results

image_paths = [
    "datasets/test/OIP.jpg",
    "datasets/test/OIP2.jpg",
    "datasets/test/OIP3.jpg",
    "datasets/test/dog.jpg",
]
results = predict(image_paths, model, inference_transform, idx_to_class, device)
for result in results:
    print(result)

加载模型
当前加载的类别信息为
{0: 'Cat', 1: 'Dog'}
模型加载成功
最佳验证准确率:  0.9895979195839167
{'path': 'datasets/test/OIP.jpg', 'class': 'Cat', 'confidence': 0.9998821020126343}
{'path': 'datasets/test/OIP2.jpg', 'class': 'Cat', 'confidence': 0.9999796152114868}
{'path': 'datasets/test/OIP3.jpg', 'class': 'Cat', 'confidence': 0.9999998807907104}
{'path': 'datasets/test/dog.jpg', 'class': 'Dog', 'confidence': 0.9999997615814209}
